# Bayes@N with EvalScope

EvalScope can report how likely a model is to answer a question from a fixed quiz correctly, along with the uncertainty in that estimate. We'll collect four answers to each of three questions, run EvalScope's built-in `bayes_at_n` aggregation, and check the report against the individual grades. The quiz and mock model both run locally. [EvalScope's evaluation FAQ](https://evalscope.readthedocs.io/en/v1.14.1/get_started/faq.html#evaluation-configuration-parameters) · [EvalScope integration PR](https://github.com/modelscope/evalscope/pull/1903)

This tutorial uses EvalScope 1.14.1. In a fresh Python 3.12 Jupyter kernel, install it once and restart the kernel.

```python
%pip install "evalscope==1.14.1"
```

## Create one local question subset

The `general_mcq` adapter reads one JSON object per line. Give each question its choices and the letter of the correct answer. Naming the file `quiz_val.jsonl` makes `quiz` the subset name for the validation split. The temporary directory holds the questions and run artifacts for this notebook session.

In [1]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory, mkdtemp

questions = [
    {
        "id": "planet", "question": "Which planet is known as the Red Planet?",
        "A": "Mars", "B": "Venus", "answer": "A",
    },
    {
        "id": "hexagon", "question": "How many sides does a hexagon have?",
        "A": "Five", "B": "Six", "answer": "B",
    },
    {
        "id": "current", "question": "Which unit measures electric current?",
        "A": "Ampere", "B": "Volt", "answer": "A",
    },
]
workspace = TemporaryDirectory()
data_dir = Path(workspace.name) / "questions"
data_dir.mkdir()
with (data_dir / "quiz_val.jsonl").open("w", encoding="utf-8") as file:
    for question in questions:
        file.write(json.dumps(question) + "\n")

## Enable Bayes@N and run the evaluation

Set `repeats` on `TaskConfig` and `aggregation` on the dataset. Select exactly one subset with a binary `accuracy` or `exact_match` grade. Here the rule-based grader checks the answer letter. With a batch size of one, the mock answers B on the first attempt and A on the remaining eleven.

In [2]:
from evalscope import TaskConfig, run_task
from evalscope.api.model import ModelOutput

repeats = 4
answers = ["B"] + ["A"] * (len(questions) * repeats - 1)
outputs = [ModelOutput.from_content(model="quiz-model", content=a) for a in answers]

In [3]:
from IPython.display import clear_output

work_dir = Path(mkdtemp(prefix="run-", dir=workspace.name))
config = TaskConfig(
    model="quiz-model",
    eval_type="mock_llm",
    model_args={"custom_outputs": outputs},
    datasets=["general_mcq"],
    dataset_args={
        "general_mcq": {
            "local_path": str(data_dir),
            "subset_list": ["quiz"],
            "aggregation": "bayes_at_n",
        }
    },
    repeats=repeats,
    eval_batch_size=1,
    judge={"strategy": "rule"},
    work_dir=str(work_dir),
    no_timestamp=True,
    collect_perf=False,
)
report = run_task(config)["general_mcq"]
clear_output()  # The full run log is in work_dir / "logs" / "eval_log.log".

## Read the report

The primary score, `bayes_at_n`, is the posterior mean. The `bayes_at_n_sd` row reports its posterior SD. Both use the 0–1 probability scale. The `n` dimension gives the attempts **per question**, and `num` counts **questions**.

In [4]:
print(f"{'aggregation':<16} {'n':>2} {'num':>5} {'score':>8}")
for metric in report.metrics:
    name = metric.identity.aggregation
    n = metric.identity.dimensions["n"]
    print(f"{name:<16} {n:>2} {metric.num:>5} {metric.score:>8.4f}")

aggregation       n   num    score
bayes_at_n        4     3   0.5556
bayes_at_n_sd     4     3   0.0891


## Check the result against the saved grades

EvalScope saves one review per attempt. Put its binary `accuracy` in the column given by `generation_index`, which starts at zero. The question ID comes from the local JSONL file.

In [5]:
import pandas as pd

review_file = work_dir / "reviews" / config.model_id / "general_mcq_quiz.jsonl"
with review_file.open(encoding="utf-8") as file:
    reviews = [json.loads(line)["sample_score"] for line in file]
outcomes = pd.json_normalize(reviews).pivot(
    index="sample_metadata.id",
    columns="generation_index",
    values="score.value.accuracy",
)
outcomes = outcomes.reindex([q["id"] for q in questions]).astype(int)
outcomes.rename_axis(index="question")

generation_index,0,1,2,3
question,,,,
planet,0,1,1,1
hexagon,0,0,0,0
current,1,1,1,1


For a question with $s$ correct answers in $N$ attempts, the independent Beta(1, 1) prior gives posterior parameters $a=s+1$ and $b=N-s+1$. The quiz mean averages the question means. To get its SD, add the question variances, take the square root, then divide by the number of questions. EvalScope keeps full-precision values in the subset metadata and rounds the report scores to four decimals.

In [6]:
from math import isclose, sqrt

correct = outcomes.sum(axis=1)
a, b = correct + 1, repeats - correct + 1
question_means = a / (a + b)
question_variances = a * b / ((a + b) ** 2 * (a + b + 1))

primary = next(
    m for m in report.metrics if m.identity == report.primary_metric_identity
)
details = primary.categories[0].subsets[0].metadata
assert isclose(details["posterior_mean"], question_means.mean())
assert isclose(details["posterior_sd"], sqrt(question_variances.sum()) / len(outcomes))

print(f"{details['questions']} questions, {details['generations']} graded answers")
print(f"Posterior mean  {details['posterior_mean']:.6f}")
print(f"Posterior SD    {details['posterior_sd']:.6f}")

3 questions, 12 graded answers
Posterior mean  0.555556
Posterior SD    0.089087


The three questions had 3, 0 and 4 correct answers. Their posterior means are $4/6$, $1/6$ and $5/6$, giving a quiz-wide estimate of **55.56%**. The posterior SD is **8.91 percentage points** for this fixed quiz.

For a model served at a local OpenAI-compatible endpoint, set `model` to the served model name, `eval_type="openai_api"` and `api_url` to its base URL. Replace the mock's `model_args` with sampling settings in `generation_config`, for example `{"temperature": 0.7}`. Keep one subset and draw independent generations under the same prompt, sampling settings and grading rule. Every question needs a usable binary score at every planned generation. Missing or duplicate trials cause the aggregation to fail.